### Data Cleaning Notes:
Dataset Sources:
- FEMA NFIP Claims of 5 Counties: Beaufort, Edgecombe, Lenoir, Robeson, Wayne
- NC Buildings GDB: Building info for 5 counties
- GeoTIFF: 7 hurricanes, 1996–2018

Output:
- aggregated grid level dataset: 57 grid cell (0.1° × 0.1°) * 7 flood events, excluding 0 flood ratio, total of 220 rows

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Paths
GDB_PATHS = {  # building info and spatial data in each county
    'Beaufort':  'Beaufort_2010_Buildings.gdb',
    'Edgecombe': 'Edgecombe_2010_Buildings.gdb',
    'Lenoir':    'Lenoir_2010_Buildings.gdb',
    'Robeson':   'Robeson_2010_Buildings.gdb',
    'Wayne':     'Wayne_2010_Buildings.gdb',
}
LAYER      = 'S_BUILDING_FP' # layer name for GDB
GRID_SIZE  = 0.1

MODEL_CSV  = 'model_dataset.csv' # claims data for each storm event per grid cell
FLOOD_CSV  = 'grid_flood_ratios.csv' # flood data for each storm event per grid cell

# output path for elevation features
OUT_ELEV = 'output/grid_elevation_features.csv'
# output path for final model dataset
OUT_MODEL = 'output/agg_model_dataset.csv'



#### NC Buildings GDB Cleaning

Feature engineering: 
Replace original `mean_elevation` with:
- **`mean_hag`** = mean(FFE − LIDAR_LAG): average height of first floor above adjacent ground
- **`pct_elevated`**: fraction of buildings with first floor >1ft above ground
These directly measure a building's physical protection against floodwater.

Output：'grid_elevation_features.csv'

In [ ]:
# Load building data from all counties
gdfs = []
for county, path in GDB_PATHS.items():
    print(f'  Loading {county}...')
    gdf = gpd.read_file(
        path, layer=LAYER,
        include_fields=['BLDG_ID', 'FFE', 'LIDAR_LAG', 'EXCLUDE', 'ISGHOST']
    )
    gdf['county_name'] = county
    gdfs.append(gdf)

bldgs = pd.concat(gdfs, ignore_index=True)
print(f'\nTotal loaded       : {len(bldgs):,}')

# Remove ghost structures and excluded records
bldgs = bldgs[(bldgs['ISGHOST'] == 0) & (bldgs['EXCLUDE'] == 0)].copy()

# Compute height above ground (FFE - LIDAR_LAG)
bldgs['FFE']       = pd.to_numeric(bldgs['FFE'],       errors='coerce')
bldgs['LIDAR_LAG'] = pd.to_numeric(bldgs['LIDAR_LAG'], errors='coerce')
bldgs['hag']       = bldgs['FFE'] - bldgs['LIDAR_LAG']

# Clip to physically plausible range (0–30 ft)
bldgs['hag'] = bldgs['hag'].where(
    (bldgs['hag'] >= 0) & (bldgs['hag'] <= 30), np.nan
)
bldgs = bldgs[bldgs['hag'].notna()]
print(f'Valid HAG values   : {len(bldgs):,}')

In [ ]:
# ── Assign buildings to 0.1° grid cells ───────────────────────────────────
bldgs_wgs     = bldgs.set_geometry('geometry').to_crs('EPSG:4326')
bldgs['lon']  = bldgs_wgs.geometry.centroid.x
bldgs['lat']  = bldgs_wgs.geometry.centroid.y

bldgs['grid_lat'] = (bldgs['lat'] / GRID_SIZE).round(0) * GRID_SIZE
bldgs['grid_lon'] = (bldgs['lon'] / GRID_SIZE).round(0) * GRID_SIZE
bldgs['grid_id']  = (
    bldgs['grid_lat'].round(1).astype(str) + '_' +
    bldgs['grid_lon'].round(1).astype(str)
)

# Keep only grids that appear in model_dataset
df_model     = pd.read_csv(MODEL_CSV)
target_grids = set(df_model['grid_id'].unique())
bldgs_study  = bldgs[bldgs['grid_id'].isin(target_grids)].copy()

print(f'Target grids          : {len(target_grids)}')
print(f'Buildings in study area: {len(bldgs_study):,}')

In [ ]:
# ── Aggregate HAG to grid level ────────────────────────────────────────────
def pct_elevated(x, threshold=1.0):
    valid = x.dropna()
    return (valid > threshold).mean() if len(valid) > 0 else np.nan

grid_elev = (
    bldgs_study
    .groupby('grid_id')['hag']
    .agg(
        mean_hag     = 'mean',
        median_hag   = 'median',
        pct_elevated = lambda x: pct_elevated(x, 1.0),
        n_bldg_hag   = 'count',
    )
    .reset_index()
    .round(4)
)
print('Comparison with replaced variables:')
print(f'  mean_elevation std (old) : {df_model["mean_elevation"].std():.3f}  ← raw sea-level')
print(f'  mean_hag       std (new) : {grid_elev["mean_hag"].std():.3f}  ← physically meaningful')
print(f'  pct_elevated   std (new) : {grid_elev["pct_elevated"].std():.3f}  ← meaningful')

In [ ]:
# ── Save elevation features ────────────────────────────────────────────────
grid_elev.to_csv(OUT_ELEV, index=False)

#### Inspect and Merge All Data Sources

In [ ]:
df_raw  = pd.read_csv(MODEL_CSV)
fr      = pd.read_csv(FLOOD_CSV)
ge      = pd.read_csv(OUT_ELEV)

In [ ]:
print('Dataset shapes:')
print(f'  model_dataset.csv   : {df_raw.shape}')
print(f'  grid_flood_ratios.csv : {fr.shape}')
print(f'  grid_elevation_features.csv : {ge.shape}')

In [ ]:
# ── Load and merge all three sources ──────────────────────────────────────
df = (
    df_raw
    .merge(fr[['grid_id', 'event_num', 'flood_ratio']], on=['grid_id', 'event_num'])
    .merge(ge[['grid_id', 'mean_hag', 'pct_elevated']],  on='grid_id')
)

print(f'model_dataset          : {df_raw.shape}')
print(f'+ flood_ratio merge    : +1 column')
print(f'+ elevation merge      : +2 columns (mean_hag, pct_elevated)')
print(f'Final shape            : {df.shape}')
print(f'Missing values         : {df.isnull().sum().sum()}')
print()
print('New variables summary:')
print(df[['flood_ratio', 'mean_hag', 'pct_elevated']].describe().round(4))

In [ ]:
df.to_csv(OUT_MODEL, index=False)

#### EDA and visualization

In [ ]:
# ── Storm-level summary ────────────────────────────────────────────────────
storm_summary = (
    df.groupby(['event_name', 'year'])
    .agg(
        grid_cells        = ('grid_id',          'count'),
        total_nfip_M      = ('total_paid',        lambda x: x.sum()/1e6),
        mean_loss_ratio   = ('loss_ratio',        'mean'),
        mean_flood_ratio  = ('flood_ratio',       'mean'),
        mean_hag_ft       = ('mean_hag',          'mean'),
        mean_pct_elevated = ('pct_elevated',      'mean'),
    )
    .round(3)
    .sort_values('year')
)
print('=== Storm-level Summary ===')
storm_summary

In [ ]:
# ── County-level summary ───────────────────────────────────────────────────
county_summary = (
    df.groupby('county')
    .agg(
        n_obs            = ('grid_id',      'count'),
        mean_loss_ratio  = ('loss_ratio',   'mean'),
        mean_hag_ft      = ('mean_hag',     'mean'),
        pct_elevated     = ('pct_elevated', 'mean'),
        total_paid_M     = ('total_paid',   lambda x: x.sum()/1e6),
    )
    .round(3)
    .sort_values('mean_loss_ratio', ascending=False)
)
print('=== County-level Summary ===')
county_summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Descriptive Statistics — Key Variables', fontsize=13)

# 1. loss_ratio distribution
axes[0,0].hist(df['loss_ratio'], bins=30, color='steelblue', edgecolor='white')
axes[0,0].axvline(df['loss_ratio'].median(), color='tomato', linestyle='--',
                  label=f'Median = {df["loss_ratio"].median():.3f}')
axes[0,0].set_xlabel('Loss Ratio'); axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Distribution of Loss Ratio')
axes[0,0].legend()

# 2. mean_hag distribution
axes[0,1].hist(df['mean_hag'], bins=25, color='seagreen', edgecolor='white')
axes[0,1].axvline(df['mean_hag'].median(), color='tomato', linestyle='--',
                  label=f'Median = {df["mean_hag"].median():.2f} ft')
axes[0,1].set_xlabel('Mean Height Above Ground (ft)'); axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Distribution of Mean HAG\n(FFE - LIDAR_LAG)')
axes[0,1].legend()

# 3. loss_ratio vs mean_hag scatter
scatter = axes[1,0].scatter(df['mean_hag'], df['loss_ratio'],
                            c=df['flood_ratio'], cmap='YlOrRd',
                            alpha=0.6, s=30)
plt.colorbar(scatter, ax=axes[1,0], label='flood_ratio')
axes[1,0].set_xlabel('Mean Height Above Ground (ft)')
axes[1,0].set_ylabel('Loss Ratio')
axes[1,0].set_title('Loss Ratio vs Height Above Ground\n(color = flood inundation ratio)')

# 4. loss_ratio by county boxplot
county_order = county_summary.index.tolist()
data_box     = [df[df['county'] == c]['loss_ratio'].values for c in county_order]
bp = axes[1,1].boxplot(data_box, labels=county_order, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue'); patch.set_alpha(0.7)
axes[1,1].set_xlabel('County'); axes[1,1].set_ylabel('Loss Ratio')
axes[1,1].set_title('Loss Ratio by County')
axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation with loss_ratio ────────────────────────────────────────────
FEATURE_COLS = [
    'flood_ratio', 'mean_hag', 'pct_elevated',
    'mean_age', 'res_ratio', 'bldg_density',
    'mean_bldg_value', 'mean_sqft', 'areaSqKm_developed', 'pct_miss_elev'
]

corr = df[FEATURE_COLS + ['loss_ratio']].corr()['loss_ratio'].drop('loss_ratio').sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tomato' if v > 0 else 'steelblue' for v in corr]
corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation with Loss Ratio (Updated Features)')
ax.set_xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

print('Correlation table:')
print(corr.round(4).to_string())